In [2]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import torch

# Load the datasets
resume_df = pd.read_csv('resume.csv')
job_df = pd.read_csv('postings.csv')

# Display basic info
print("Resume Dataset Info:")
print(resume_df.info())
print("\nJob Postings Dataset Info:")
print(job_df.info())

# Show first few rows
print("\nResume Dataset Sample:")
display(resume_df.head())
print("\nJob Postings Dataset Sample:")
display(job_df.head())

Resume Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB
None

Job Postings Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period           

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR



Job Postings Dataset Sample:


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [3]:
def preprocess_text(text):
    """Basic text preprocessing"""
    if pd.isnull(text):
        return ""
    # Convert to lowercase
    text = text.lower()
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s,.!?-]', '', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Preprocess resume text
print("Preprocessing resume text...")
resume_df['processed_text'] = resume_df['Resume_str'].apply(preprocess_text)

# Preprocess job text
print("Preprocessing job descriptions...")
job_df['processed_title'] = job_df['title'].apply(preprocess_text)
job_df['processed_description'] = job_df['description'].apply(preprocess_text)
job_df['processed_skills'] = job_df['skills_desc'].apply(preprocess_text)

# Display processed samples
print("\nProcessed Resume Sample:")
display(resume_df[['ID', 'processed_text']].head())
print("\nProcessed Job Sample:")
display(job_df[['job_id', 'processed_title', 'processed_description', 'processed_skills']].head())

Preprocessing resume text...
Preprocessing job descriptions...

Processed Resume Sample:


,ID,processed_text
0,16852973,hr administratormarketing associate hr adminis...
1,22323967,"hr specialist, us hr operations summary versat..."
2,33176873,hr director summary over 20 years experience i...
3,27018550,"hr specialist summary dedicated, driven, and d..."
4,17812897,hr manager skill highlights hr skills hr depar...



Processed Job Sample:


,job_id,processed_title,processed_description,processed_skills
0,921716,marketing coordinator,job descriptiona leading real estate firm in n...,requirements we are seeking a college or gradu...
1,1829192,mental health therapistcounselor,"at aspen therapy and wellness , we are committ...",
2,10998357,assitant restaurant manager,the national exemplar is accepting application...,we are currently accepting resumes for foh - a...
3,23221523,senior elder law trusts and estates associate ...,senior associate attorney - elder law trusts a...,this position requires a baseline understandin...
4,35982263,service technician,looking for hvac service tech with experience ...,


In [4]:
# Combine job features into a single text
print("Combining job features...")
job_df['combined_text'] = job_df.apply(
    lambda row: f"Title: {row['processed_title']}\nDescription: {row['processed_description']}\nSkills: {row['processed_skills']}",
    axis=1
)

# Display combined text sample
print("\nCombined Job Text Sample:")
display(job_df[['job_id', 'combined_text']].head())

Combining job features...

Combined Job Text Sample:


,job_id,combined_text
0,921716,Title: marketing coordinator\nDescription: job...
1,1829192,Title: mental health therapistcounselor\nDescr...
2,10998357,Title: assitant restaurant manager\nDescriptio...
3,23221523,Title: senior elder law trusts and estates ass...
4,35982263,Title: service technician\nDescription: lookin...


In [ ]:
# Save processed data
print("Saving processed data...")
resume_df.to_csv('processed_resumes.csv', index=False)
job_df.to_csv('processed_jobs.csv', index=False)

print("Processed data saved successfully!")

Saving processed data...
